# 00 · The inference layer
**5 minutes · One request, its measurements, and streaming**

We use two prepared methods throughout the talk: `generate` and `stream`.
The model ID selects OpenAI Responses or Bedrock Converse automatically.
Our focus is what the calls do and how we measure them.

**The setup cell** reads the model and AWS settings, loads matching prices
when available, and prints the chosen API route. It prepares configuration;
the first model request happens in the following section.

## Install requirements

Install the notebook dependencies into the active notebook kernel, then restart the kernel before continuing. If this environment has no `pip`, the cell prepares it first.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pip") is None:
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])

%pip install -r requirements.txt

In [ ]:
from IPython.display import display
from utils.workshop import Rates, Run, Settings, compare

settings = Settings.from_env()
rates = Rates.load(model_id=settings.model_id)
print(f"{settings.model_id} → {settings.resolved_api}")

## Edit the prompts

`system_prompt` controls model behavior; an empty string leaves it unspecified. `instruction` and `stream_instruction` supply the tasks for the two examples.

Rerun this cell after edits, then rerun the relevant example. For comparisons, rerun both baseline and tuned versions.

In [ ]:
system_prompt = ""
instruction = "Explain why an agent can need several model calls, in two sentences."
stream_instruction = "Explain why repeated context adds token cost, in two sentences."

## Make one request

The `with Run(...)` block measures one complete task. Inside it, `generate`
sends the prompt and returns a response whose visible answer is in `.text`.
`max_output_tokens=2048` is an upper limit, including any reasoning tokens;
it is not a request to generate exactly that many tokens.

After the timed block, we print the answer, display the measurements, and
save the trace for later comparison.

In [ ]:
with Run("00_baseline", settings, rates) as baseline_run:
    response = baseline_run.generate(
        "answer",
        instruction,
        instructions=system_prompt,
        max_output_tokens=2048,
    )

print(response.text)
display(baseline_run.metrics())
display(compare(baseline_run))
baseline_run.save()

**Read the two tables:** `metrics()` shows each attempt; `compare()` shows
totals for the completed task.

- **Input / Output:** tokens reported by the model. Reported reasoning
  tokens are already included in Output.
- **Seconds / Retry wait:** client-observed attempt time and any separate
  wait before retrying. Both contribute to the task's total elapsed time.
- **Reported USD:** estimated cost of reported model usage. Missing prices
  or usage remain unavailable rather than becoming zero.

The default policy permits three attempts for retryable failures. A real
failed request can have unreported usage; check **Usage complete**.
`baseline_run.frame()` retains additional details such as cache counts.

## Stream the same way

With `RUN_STREAMING=True`, this cell makes one additional request.
`print_fragment` receives visible text chunks and prints them immediately.
`stream()` returns after consuming the full response, including final usage.

**Watch two timings:** the first visible text and total completion time.
Streaming can make the answer feel more responsive without reducing its
token count. Request-establishment failures can retry; a failure after the
stream opens is surfaced without silently replaying partial output.

In [ ]:
RUN_STREAMING = True

def print_fragment(text):
    print(text, end="", flush=True)

if RUN_STREAMING:
    with Run("00_streaming", settings, rates) as stream_run:
        streamed = stream_run.stream(
            "answer",
            stream_instruction,
            instructions=system_prompt,
            max_output_tokens=2048,
            on_text=print_fragment,
        )
    print()
    print("First text (s):", stream_run.attempts[-1].time_to_first_text_s)
    display(stream_run.metrics())

**Next:** the same measurements will follow an entire tool round trip,
a three-call chain, and a retrieval-plus-generation task. Each example
compares a working baseline with one deliberate tuning change.